# Healthcare Pharma Data Pipeline
# Notebook 04 - Silver Layer

In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_tables = {
    "biotech_bronze": "biotech_funding_silver",
    "clinical_trials_bronze": "clinical_trials_silver",
    "disease_burden_bronze": "disease_burden_silver",
    "drug_approvals_bronze": "drug_approvals_silver",
    "pharma_companies_bronze": "pharma_companies_silver"
}

In [0]:
for bronze_table, silver_table in bronze_tables.items():

    print("="*60)
    print(f"Processing {bronze_table}")

    df = spark.table(f"workspace.default.{bronze_table}")

    # Remove duplicate rows
    df = df.dropDuplicates()

    # Remove rows where every column is null
    df = df.na.drop(how="all")

    # Standardize column names
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower().replace(" ", "_"))

    # Trim spaces from string columns
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]

    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))

    # Save Silver Table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.default.{silver_table}")

    print(f"Saved {silver_table}")

Processing biotech_bronze
Saved biotech_funding_silver
Processing clinical_trials_bronze
Saved clinical_trials_silver
Processing disease_burden_bronze
Saved disease_burden_silver
Processing drug_approvals_bronze
Saved drug_approvals_silver
Processing pharma_companies_bronze
Saved pharma_companies_silver


In [0]:
%sql

SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,biotech_bronze,false
default,biotech_funding_silver,false
default,clinical_trials_bronze,false
default,clinical_trials_silver,false
default,disease_burden_bronze,false
default,disease_burden_silver,false
default,drug_approvals_bronze,false
default,drug_approvals_silver,false
default,pharma_companies_bronze,false
default,pharma_companies_silver,false


In [0]:
silver_tables = [
    "biotech_funding_silver",
    "clinical_trials_silver",
    "disease_burden_silver",
    "drug_approvals_silver",
    "pharma_companies_silver"
]

for table in silver_tables:
    count = spark.table(f"workspace.default.{table}").count()
    print(f"{table} : {count}")

biotech_funding_silver : 1208
clinical_trials_silver : 599
disease_burden_silver : 3310
drug_approvals_silver : 722
pharma_companies_silver : 489
